## Imports

In [1]:
import os
import random
import torch
import wandb
import numpy as np

from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR

from fairface_vit       import FairFaceViT
from dataset.dataloader import get_dataset, get_dataloaders
from dataset.transforms import get_train_transform, get_age_transform, get_val_transform
from training.trainer   import train_loop, test_loop
from training.losses    import get_age_weights, get_race_weights, get_loss_function

/home/ashkanrn/01-Project/University/Deep-Learning/DL-project/.venv/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Seed

In [2]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## Model selection

In [3]:
# change ONLY this to switch models
model_name = "clip"  # "clip" | "dinov2" | "siglip"

if model_name == "clip":
    from model.clip import model as backbone_model, processor as backbone_processor

elif model_name == "dinov2":
    from model.dinov2 import model as backbone_model, processor as backbone_processor

elif model_name == "siglip":
    from model.siglip import model as backbone_model, processor as backbone_processor

else:
    raise ValueError(f"Unknown model_name: {model_name}")

RUN_NAMES = {
    "clip"  : "clip-vit-base-patch16",
    "dinov2": "dinov2-vit-base",
    "siglip": "siglip-vit-base-patch16",
}

# for wandb
run_name = RUN_NAMES[model_name]
run_version = 'v2'

checkpoint_dir  = f"../checkpoints/{model_name}/{run_version}"
checkpoint_path = f"{checkpoint_dir}/{model_name}-checkpoint.pt"
best_heads_path = f"{checkpoint_dir}/{model_name}-best-heads.pt"
os.makedirs(checkpoint_dir, exist_ok=True)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 26122.51it/s]
[transformers] CLIPVisionModel LOAD REPORT from: ../models/clip-vit-base-patch16
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias  

## Resume/fresh control

In [4]:
#set this yourself
mode = "fresh"   # "auto" | "fresh" | "resume"

if mode == "fresh":
    resume_run = False
elif mode == "resume":
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"mode='resume' but no checkpoint at {checkpoint_path}")
    resume_run = True
else:  # "auto"
    resume_run = os.path.exists(checkpoint_path)

## Hyperparameters

In [5]:
epochs         = 20
batch_size     = 20

learning_rate  = 1e-4
warmup_epochs  = 2  

weight_decay   = 1e-2

age_dropout1   = 0.15
age_hidden_dim = 384

LOSS_WEIGHT = {
    "gender": 1,
    "age":    1,
    "race":   1
}
TASK_WEIGHTS = {
    "gender": 0.25,
    "race":   0.35,
    "age":    0.40,
}

## WandB

"allow" handles both fresh start and resume via the same id

In [6]:
# run_id = f"{run_name}-{run_version}"

# wandb.init(
#     project="fairface-vit",
#     name=run_id,
#     id=run_id,
#     resume="allow",
#     mode="offline",
#     config={
#         "epochs": epochs,
#         "batch_size": batch_size,
#         "learning_rate": learning_rate,
#         "age_dropout1": age_dropout1,
#         "age_hidden_dim": age_hidden_dim,
#         "weight_decay": weight_decay,
#         "optimizer": "AdamW",
#         "model": run_name,
#         "loss_weights": LOSS_WEIGHT,
#     },
# )
# print(wandb.run)

## Model and device

In [7]:

fairface_model = FairFaceViT(backbone_model, age_dropout1, age_hidden_dim)
device = "cuda" if torch.cuda.is_available() else "cpu"
fairface_model.to(device)

FairFaceViT(
  (backbone): CLIPVisionModel(
    (embeddings): CLIPVisionEmbeddings(
      (patch_embedding): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), bias=False)
      (position_embedding): Embedding(197, 768)
    )
    (pre_layrnorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)

## Dataset and dataloaders

In [8]:
train_set, val_set = get_dataset(
    get_train_transform(backbone_processor),
    get_age_transform(backbone_processor),
    get_val_transform(backbone_processor),
)

train_dataloader, val_dataloader = get_dataloaders(
    train_set,
    val_set,
    batch_size=batch_size,
    num_workers=0,
)

## Loss functions

In [9]:
age_labels = np.array(train_set.dataset["age"])
race_labels = np.array(train_set.dataset["race"])

age_weights = get_age_weights(age_labels, num_classes=9, device=device)
race_weights = get_race_weights(race_labels, num_classes=7, device=device)

loss_funcs = get_loss_function(age_weights, race_weights)

## Optimizer

In [10]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, fairface_model.parameters()),
    lr=learning_rate,
    weight_decay=weight_decay,
)

## LR Scheduler

In [11]:
warmup_scheduler = LinearLR(
    optimizer,
    start_factor=0.1,
    end_factor=1.0,
    total_iters=warmup_epochs
)

cosine_scheduler = CosineAnnealingLR(
    optimizer,
    T_max=epochs - warmup_epochs,
    eta_min=1e-6
)

scheduler = SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[warmup_epochs]
)

## Resume state

In [12]:
best_score         = 0
patience         = 6
patience_counter = 0
min_delta        = 0.001
start_epoch      = 0

In [13]:
if resume_run:
    checkpoint = torch.load(
        checkpoint_path, 
        map_location=device
    )
    print(checkpoint.keys())
    print(checkpoint["epoch"])

    fairface_model.gender.load_state_dict(
        checkpoint["gender_head"]
    )
    fairface_model.age.load_state_dict(
        checkpoint["age_head"]
    )
    fairface_model.race.load_state_dict(
        checkpoint["race_head"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    scheduler.load_state_dict(         
        checkpoint["scheduler_state_dict"]
    )
    
    for state in optimizer.state.values():
        for key, value in state.items():
            if torch.is_tensor(value):
                state[key] = value.to(device)

    best_score         = checkpoint["best_acc"]
    start_epoch      = checkpoint["epoch"] + 1
    patience         = checkpoint["patience"]
    patience_counter = checkpoint["patience_counter"]
    min_delta        = checkpoint["min_delta"]

    torch.set_rng_state(
        checkpoint["torch_rng_state"]
    )
    if torch.cuda.is_available() and checkpoint["cuda_rng_state"] is not None:
        torch.cuda.set_rng_state_all(
            checkpoint["cuda_rng_state"]
        )

    np.random.set_state(
        checkpoint["numpy_rng_state"]
    )
    
    random.setstate(
        checkpoint["python_rng_state"]
    )

    print(
        f"Loaded {model_name}-checkpoint.pt | "
        f"Resume from epoch {start_epoch} | "
        f"Best acc={best_score:.4f}"
    )
else:
    print(f"No checkpoint found at {checkpoint_path} — starting fresh.")

No checkpoint found at ../checkpoints/clip/v2/clip-checkpoint.pt — starting fresh.


## Training

In [ ]:
for epoch in range(start_epoch, epochs):

    print(f"\nEpoch {epoch+1}/{epochs}")

    train_loss, train_task_loss, train_metrics = train_loop(
        train_dataloader, fairface_model, loss_funcs,
        LOSS_WEIGHT, optimizer, device, epoch + 1, epochs
    )

    val_loss, val_task_loss, metrics, subgroup_metrics = test_loop(
        val_dataloader, fairface_model, loss_funcs,
        LOSS_WEIGHT, device, epoch + 1, epochs
    )

    scheduler.step() 

    log_dict = {
        "epoch": epoch + 1,
        "lr": optimizer.param_groups[0]["lr"],

        "train/loss": train_loss,
        "val/loss": val_loss,

        "train/gender_loss": train_task_loss["gender"],
        "train/age_loss": train_task_loss["age"],
        "train/race_loss": train_task_loss["race"],

        "val/gender_loss": val_task_loss["gender"],
        "val/age_loss": val_task_loss["age"],
        "val/race_loss": val_task_loss["race"],
    }

    for task, values in train_metrics.items():
        for metric_name, value in values.items():
            log_dict[f"train/{task}/{metric_name}"] = value

    for task, values in metrics.items():
        for metric_name, value in values.items():
            log_dict[f"val/{task}/{metric_name}"] = value

    for group_name, values in subgroup_metrics.items():
        for subgroup, acc in values.items():
            log_dict[f"subgroup/{group_name}/{subgroup}"] = acc



    avg_accuracy = (
        metrics["gender"]["accuracy"]
        + 
        metrics["age"]["accuracy"]
        + 
        metrics["race"]["accuracy"]
    ) / 3


    current_score = (
        TASK_WEIGHTS["gender"] * metrics["gender"]["f1"]
        + 
        TASK_WEIGHTS["race"] * metrics["race"]["f1"]
        +  
        TASK_WEIGHTS["age"] * metrics["age"]["f1"]
    )

    log_dict["val/avg_accuracy"] = avg_accuracy
    log_dict["val/weighted_f1_score"] = current_score

    wandb.log(log_dict)

    if current_score > best_score + min_delta:

        best_score = current_score
        patience_counter = 0

        torch.save(
            {
                "gender_head": fairface_model.gender.state_dict(),
                "age_head":    fairface_model.age.state_dict(),
                "race_head":   fairface_model.race.state_dict(),
                
                "epoch":       epoch,       
                "best_acc":    best_score, 
            },
            best_heads_path,
        )

        print(f"Saved {model_name}-best-heads.pt | weighted F1 score={best_score:.4f}")

    else:
        patience_counter += 1
        print(f"No improvement ({patience_counter}/{patience}), {best_score:.4f}")

    torch.save(
        {
            "gender_head": fairface_model.gender.state_dict(),
            "age_head":    fairface_model.age.state_dict(),
            "race_head":   fairface_model.race.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),

            "epoch":    epoch,
            "best_acc": best_score,

            "patience":         patience,
            "patience_counter": patience_counter,
            "min_delta":        min_delta,

            "python_rng_state": random.getstate(),
            "numpy_rng_state":  np.random.get_state(),
            "torch_rng_state":  torch.get_rng_state(),
            "cuda_rng_state": (
                torch.cuda.get_rng_state_all()
                if torch.cuda.is_available()
                else None
            ),
        },
        checkpoint_path,
    )

    if patience_counter >= patience:
        print(f"\nEarly stopping after {epoch+1} epochs.")
        break


Epoch 1/20


KeyboardInterrupt: 

In [ ]:
wandb.finish()